In [22]:
import os 
import numpy as np 
import tensorflow as tf 

SEED = 42
tf.keras.utils.set_random_seed(SEED)
np.random.seed(SEED)
print(tf.__version__)

2.18.0


In [23]:
# load MNIST and preprocess
from tensorflow.keras.datasets import mnist

(x_train_full, y_train_full), (x_test, y_test) = mnist.load_data()

# scale to [0, 1]
x_train_full = x_train_full.astype('float32') / 255.0
x_test = x_test.astype('float32') / 255.0

# CNN expect (28, 28, 1)
x_train_full = np.expand_dims(x_train_full, -1)
x_test = np.expand_dims(x_test, -1)

num_classes = 10

# split train/validation set (55k/5k)
val_size = 5000
x_val = x_train_full[-val_size:]
y_val = y_train_full[-val_size:]
x_train = x_train_full[:-val_size]
y_train = y_train_full[:-val_size]

print("Train:", x_train.shape, "Val:", x_val.shape, "Test:", x_test.shape)

# one-hot encoding
y_train_oh = tf.keras.utils.to_categorical(y_train, num_classes)
y_val_oh = tf.keras.utils.to_categorical(y_val, num_classes)
y_test_oh = tf.keras.utils.to_categorical(y_test, num_classes)

Train: (55000, 28, 28, 1) Val: (5000, 28, 28, 1) Test: (10000, 28, 28, 1)


In [24]:
# data augmentation
augment = tf.keras.Sequential([
    tf.keras.layers.RandomRotation(0.08),
    tf.keras.layers.RandomTranslation(0.08, 0.08),
    tf.keras.layers.RandomZoom(0.10),
], name="augmentation")

hard_augment = tf.keras.Sequential([
    tf.keras.layers.RandomRotation(0.10),
    tf.keras.layers.RandomTranslation(0.12, 0.12),
    tf.keras.layers.RandomZoom(0.12),
], name="hard_augmentation")


In [25]:
# Model
L2 = tf.keras.regularizers.l2(1e-4)

def conv_bn_act(x, filters, k=3, s=1):
    x = tf.keras.layers.Conv2D(filters, k, strides=s, padding='same',
                               use_bias=False, kernel_regularizer=L2)(x)
    x = tf.keras.layers.BatchNormalization()(x)
    x = tf.keras.layers.Activation('relu')(x)
    return x

def residual_block(x, filters, downsample=False):
    s = 2 if downsample else 1
    shortcut = x

    x = conv_bn_act(x, filters, k=3, s=s)
    x = tf.keras.layers.Conv2D(filters, 3, strides=1, padding='same',
                               use_bias=False, kernel_regularizer=L2)(x)
    x = tf.keras.layers.BatchNormalization()(x)

    if downsample or shortcut.shape[-1] != filters:
        shortcut = tf.keras.layers.Conv2D(filters, 1, strides=s, padding="same",
                                          use_bias=False, kernel_regularizer=L2)(shortcut)
        shortcut = tf.keras.layers.BatchNormalization()(shortcut)

    x = tf.keras.layers.Add()([x, shortcut])
    x = tf.keras.layers.Activation('relu')(x)
    return x

inputs = tf.keras.Input(shape=(28, 28, 1))
x = augment(inputs)

x = conv_bn_act(x, 32)
x = residual_block(x, 32)
x = residual_block(x, 32)

x = residual_block(x, 64, downsample=True)
x = residual_block(x, 64)

x = residual_block(x, 128, downsample=True)
x = residual_block(x, 128)

x = tf.keras.layers.GlobalAveragePooling2D()(x)
x = tf.keras.layers.Dropout(0.4)(x)
outputs = tf.keras.layers.Dense(num_classes, activation='softmax')(x)

model = tf.keras.Model(inputs, outputs)
model.summary()

Model: "functional_3"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ input_layer_2       │ (None, 28, 28, 1) │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ augmentation        │ (None, 28, 28, 1) │          0 │ input_layer_2[0]… │
│ (Sequential)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2d_15 (Conv2D)  │ (None, 28, 28,    │        288 │ augmentation[0][… │
│                     │ 32)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ batch_normalizatio… │ (None, 28, 28,    │        128 │ conv2d_15[0][0]   │
│ (BatchNormalizatio… │ 32)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ activation_13       │ (None, 28, 28,    │          0 │ batch_normalizat… │
│ (Activation)        │ 32)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2d_16 (Conv2D)  │ (None, 28, 28,    │      9,216 │ activation_13[0]… │
│                     │ 32)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ batch_normalizatio… │ (None, 28, 28,    │        128 │ conv2d_16[0][0]   │
│ (BatchNormalizatio… │ 32)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ activation_14       │ (None, 28, 28,    │          0 │ batch_normalizat… │
│ (Activation)        │ 32)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2d_17 (Conv2D)  │ (None, 28, 28,    │      9,216 │ activation_14[0]… │
│                     │ 32)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ batch_normalizatio… │ (None, 28, 28,    │        128 │ conv2d_17[0][0]   │
│ (BatchNormalizatio… │ 32)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ add_6 (Add)         │ (None, 28, 28,    │          0 │ batch_normalizat… │
│                     │ 32)               │            │ activation_13[0]… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ activation_15       │ (None, 28, 28,    │          0 │ add_6[0][0]       │
│ (Activation)        │ 32)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2d_18 (Conv2D)  │ (None, 28, 28,    │      9,216 │ activation_15[0]… │
│                     │ 32)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ batch_normalizatio… │ (None, 28, 28,    │        128 │ conv2d_18[0][0]   │
│ (BatchNormalizatio… │ 32)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ activation_16       │ (None, 28, 28,    │          0 │ batch_normalizat… │
│ (Activation)        │ 32)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2d_19 (Conv2D)  │ (None, 28, 28,    │      9,216 │ activation_16[0]… │
│                     │ 32)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ batch_normalizatio… │ (None, 28, 28,    │        128 │ conv2d_19[0][0] 

 Total params: 698,282 (2.66 MB)

 Trainable params: 696,042 (2.66 MB)

 Non-trainable params: 2,240 (8.75 KB)

In [26]:
# compile
loss = tf.keras.losses.CategoricalCrossentropy(label_smoothing=0.05)

steps_per_epoch = int(np.ceil(x_train.shape[0] / 128))
total_steps = steps_per_epoch * 60
lr_schedule = tf.keras.optimizers.schedules.CosineDecay(
    initial_learning_rate=1e-3,
    decay_steps=total_steps,
    alpha=1e-2
)


model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=lr_schedule),
    loss=loss,
    metrics=["accuracy"]
)

In [27]:
class HardValCallback(tf.keras.callbacks.Callback):
    def __init__(self, x_val, y_val_oh, hard_aug, batch_size=256):
        super().__init__()
        self.x_val = x_val
        self.y_val = y_val_oh
        self.hard_aug = hard_aug
        self.batch_size = batch_size
        self.best_hard = -1.0

    def on_epoch_end(self, epoch, logs=None):
        # Apply hard augmentation ONCE to build a "hard" copy of validation
        xh = self.hard_aug(self.x_val, training=True).numpy()
        preds = self.model.predict(xh, batch_size=self.batch_size, verbose=0)
        hard_acc = (np.argmax(preds, axis=1) == np.argmax(self.y_val, axis=1)).mean()
        logs = logs or {}
        logs["hard_val_accuracy"] = hard_acc
        if hard_acc > self.best_hard:
            self.best_hard = hard_acc
        print(f" - hard_val_accuracy: {hard_acc:.4f} (best: {self.best_hard:.4f})")


In [28]:
ckpt_path = "cw1_best_model.h5"
callbacks = [
    HardValCallback(x_val, y_val_oh, hard_augment),
    tf.keras.callbacks.ModelCheckpoint(
        ckpt_path, monitor="hard_val_accuracy", mode="max",
        save_best_only=True, save_weights_only=False, verbose=1
    ),
    tf.keras.callbacks.EarlyStopping(
        monitor="hard_val_accuracy", mode="max", patience=12,
        restore_best_weights=True, verbose=1
    ),
]


In [29]:
# train
history = model.fit(
    x_train, y_train_oh,
    validation_data=(x_val, y_val_oh),
    epochs=60,
    batch_size=128,
    callbacks=callbacks,
    verbose=2
)

# evalute
test_loss, test_acc = model.evaluate(x_test, y_test_oh, verbose=0)
print(f"MNIST test accuracy: {test_acc*100:.2f}%")

print("Saved best model:", os.path.abspath(ckpt_path))

Epoch 1/60
 - hard_val_accuracy: 0.3826 (best: 0.3826)

Epoch 1: hard_val_accuracy improved from None to 0.38260, saving model to cw1_best_model.h5



Epoch 1: finished saving model to cw1_best_model.h5
430/430 - 88s - 204ms/step - accuracy: 0.9284 - loss: 0.6276 - val_accuracy: 0.7406 - val_loss: 1.0118 - hard_val_accuracy: 0.3826
Epoch 2/60
 - hard_val_accuracy: 0.9782 (best: 0.9782)

Epoch 2: hard_val_accuracy improved from 0.38260 to 0.97820, saving model to cw1_best_model.h5



Epoch 2: finished saving model to cw1_best_model.h5
430/430 - 84s - 196ms/step - accuracy: 0.9809 - loss: 0.4618 - val_accuracy: 0.9828 - val_loss: 0.4491 - hard_val_accuracy: 0.9782
Epoch 3/60
 - hard_val_accuracy: 0.9624 (best: 0.9782)

Epoch 3: hard_val_accuracy did not improve from 0.97820
430/430 - 85s - 197ms/step - accuracy: 0.9851 - loss: 0.4335 - val_accuracy: 0.9818 - val_loss: 0.4475 - hard_val_accuracy: 0.9624
Epoch 4/60
 - hard_val_accuracy: 0.9774 (best: 0.9782)

Epoch 4: hard_val_accuracy did not improve from 0.97820
430/430 - 84s - 194ms/step - accuracy: 0.9851 - loss: 0.4185 - val_accuracy: 0.9848 - val_loss: 0.4029 - hard_val_accuracy: 0.9774
Epoch 5/60
 - hard_val_accuracy: 0.9838 (best: 0.9838)

Epoch 5: hard_val_accuracy improved from 0.97820 to 0.98380, saving model to cw1_best_model.h5



Epoch 5: finished saving model to cw1_best_model.h5
430/430 - 83s - 194ms/step - accuracy: 0.9866 - loss: 0.4066 - val_accuracy: 0.9894 - val_loss: 0.3823 - hard_val_accuracy: 0.9838
Epoch 6/60
 - hard_val_accuracy: 0.9832 (best: 0.9838)

Epoch 6: hard_val_accuracy did not improve from 0.98380
430/430 - 84s - 196ms/step - accuracy: 0.9877 - loss: 0.3969 - val_accuracy: 0.9888 - val_loss: 0.3792 - hard_val_accuracy: 0.9832
Epoch 7/60
 - hard_val_accuracy: 0.9808 (best: 0.9838)

Epoch 7: hard_val_accuracy did not improve from 0.98380
430/430 - 83s - 194ms/step - accuracy: 0.9881 - loss: 0.3912 - val_accuracy: 0.9916 - val_loss: 0.3646 - hard_val_accuracy: 0.9808
Epoch 8/60
 - hard_val_accuracy: 0.9590 (best: 0.9838)

Epoch 8: hard_val_accuracy did not improve from 0.98380
430/430 - 83s - 194ms/step - accuracy: 0.9886 - loss: 0.3868 - val_accuracy: 0.9798 - val_loss: 0.3999 - hard_val_accuracy: 0.9590
Epoch 9/60
 - hard_val_accuracy: 0.9814 (best: 0.9838)

Epoch 9: hard_val_accuracy did 


Epoch 11: finished saving model to cw1_best_model.h5
430/430 - 83s - 194ms/step - accuracy: 0.9893 - loss: 0.3773 - val_accuracy: 0.9908 - val_loss: 0.3610 - hard_val_accuracy: 0.9842
Epoch 12/60
 - hard_val_accuracy: 0.9888 (best: 0.9888)

Epoch 12: hard_val_accuracy improved from 0.98420 to 0.98880, saving model to cw1_best_model.h5


...


Epoch 33: finished saving model to cw1_best_model.h5
430/430 - 86s - 199ms/step - accuracy: 0.9946 - loss: 0.3368 - val_accuracy: 0.9894 - val_loss: 0.3351 - hard_val_accuracy: 0.9932
Epoch 34/60
 - hard_val_accuracy: 0.9926 (best: 0.9932)

Epoch 34: hard_val_accuracy did not improve from 0.99320
430/430 - 83s - 194ms/step - accuracy: 0.9945 - loss: 0.3357 - val_accuracy: 0.9956 - val_loss: 0.3202 - hard_val_accuracy: 0.9926
Epoch 35/60
 - hard_val_accuracy: 0.9904 (best: 0.9932)

Epoch 35: hard_val_accuracy did not improve from 0.99320
430/430 - 83s - 193ms/step - accuracy: 0.9953 - loss: 0.3331 - val_accuracy: 0.9920 - val_loss: 0.3289 - hard_val_accuracy: 0.9904
Epoch 36/60
 - hard_val_accuracy: 0.9928 (best: 0.9932)

Epoch 36: hard_val_accuracy did not improve from 0.99320
430/430 - 83s - 193ms/step - accuracy: 0.9954 - loss: 0.3315 - val_accuracy: 0.9948 - val_loss: 0.3194 - hard_val_accuracy: 0.9928
Epoch 37/60
 - hard_val_accuracy: 0.9934 (best: 0.9934)

Epoch 37: hard_val_accu


Epoch 37: finished saving model to cw1_best_model.h5
430/430 - 83s - 193ms/step - accuracy: 0.9957 - loss: 0.3299 - val_accuracy: 0.9956 - val_loss: 0.3172 - hard_val_accuracy: 0.9934
Epoch 38/60
 - hard_val_accuracy: 0.9922 (best: 0.9934)

Epoch 38: hard_val_accuracy did not improve from 0.99340
430/430 - 84s - 196ms/step - accuracy: 0.9955 - loss: 0.3296 - val_accuracy: 0.9930 - val_loss: 0.3244 - hard_val_accuracy: 0.9922
Epoch 39/60
 - hard_val_accuracy: 0.9950 (best: 0.9950)

Epoch 39: hard_val_accuracy improved from 0.99340 to 0.99500, saving model to cw1_best_model.h5



Epoch 39: finished saving model to cw1_best_model.h5
430/430 - 84s - 196ms/step - accuracy: 0.9961 - loss: 0.3274 - val_accuracy: 0.9940 - val_loss: 0.3182 - hard_val_accuracy: 0.9950
Epoch 40/60
 - hard_val_accuracy: 0.9948 (best: 0.9950)

Epoch 40: hard_val_accuracy did not improve from 0.99500
430/430 - 89s - 208ms/step - accuracy: 0.9960 - loss: 0.3268 - val_accuracy: 0.9954 - val_loss: 0.3168 - hard_val_accuracy: 0.9948
Epoch 41/60
 - hard_val_accuracy: 0.9934 (best: 0.9950)

Epoch 41: hard_val_accuracy did not improve from 0.99500
430/430 - 89s - 207ms/step - accuracy: 0.9963 - loss: 0.3260 - val_accuracy: 0.9960 - val_loss: 0.3175 - hard_val_accuracy: 0.9934
Epoch 42/60
 - hard_val_accuracy: 0.9932 (best: 0.9950)

Epoch 42: hard_val_accuracy did not improve from 0.99500
430/430 - 86s - 200ms/step - accuracy: 0.9965 - loss: 0.3241 - val_accuracy: 0.9950 - val_loss: 0.3153 - hard_val_accuracy: 0.9932
Epoch 43/60
 - hard_val_accuracy: 0.9920 (best: 0.9950)

Epoch 43: hard_val_accu


Epoch 49: finished saving model to cw1_best_model.h5
430/430 - 85s - 197ms/step - accuracy: 0.9980 - loss: 0.3169 - val_accuracy: 0.9956 - val_loss: 0.3093 - hard_val_accuracy: 0.9956
Epoch 50/60
 - hard_val_accuracy: 0.9948 (best: 0.9956)

Epoch 50: hard_val_accuracy did not improve from 0.99560
430/430 - 85s - 197ms/step - accuracy: 0.9985 - loss: 0.3160 - val_accuracy: 0.9968 - val_loss: 0.3084 - hard_val_accuracy: 0.9948
Epoch 51/60
 - hard_val_accuracy: 0.9948 (best: 0.9956)

Epoch 51: hard_val_accuracy did not improve from 0.99560
430/430 - 83s - 193ms/step - accuracy: 0.9981 - loss: 0.3162 - val_accuracy: 0.9956 - val_loss: 0.3101 - hard_val_accuracy: 0.9948
Epoch 52/60
 - hard_val_accuracy: 0.9954 (best: 0.9956)

Epoch 52: hard_val_accuracy did not improve from 0.99560
430/430 - 83s - 194ms/step - accuracy: 0.9986 - loss: 0.3151 - val_accuracy: 0.9954 - val_loss: 0.3084 - hard_val_accuracy: 0.9954
Epoch 53/60
 - hard_val_accuracy: 0.9952 (best: 0.9956)

Epoch 53: hard_val_accu


Epoch 58: finished saving model to cw1_best_model.h5
430/430 - 86s - 199ms/step - accuracy: 0.9989 - loss: 0.3124 - val_accuracy: 0.9964 - val_loss: 0.3077 - hard_val_accuracy: 0.9958
Epoch 59/60
 - hard_val_accuracy: 0.9946 (best: 0.9958)

Epoch 59: hard_val_accuracy did not improve from 0.99580
430/430 - 84s - 196ms/step - accuracy: 0.9991 - loss: 0.3121 - val_accuracy: 0.9964 - val_loss: 0.3070 - hard_val_accuracy: 0.9946
Epoch 60/60
 - hard_val_accuracy: 0.9958 (best: 0.9958)

Epoch 60: hard_val_accuracy did not improve from 0.99580
430/430 - 84s - 196ms/step - accuracy: 0.9989 - loss: 0.3124 - val_accuracy: 0.9966 - val_loss: 0.3070 - hard_val_accuracy: 0.9958
Restoring model weights from the end of the best epoch: 58.
MNIST test accuracy: 99.64%
Saved best model: /Users/amir/Desktop/Pattern CW_1/cw1_best_model.h5


In [30]:
# verification

from tensorflow.keras.models import load_model
print("TensorFlow version:", tf.__version__)

# Load saved model
model = load_model("cw1_best_model.h5")

print("\nModel loaded successfully.")
model.summary()

# Load MNIST test dataset
(_, _), (x_test, y_test) = mnist.load_data()

# Preprocess exactly as training
x_test = x_test.astype("float32") / 255.0
x_test = np.expand_dims(x_test, -1)

# One-hot encode labels
y_test_oh = tf.keras.utils.to_categorical(y_test, 10)

# Evaluate
loss, accuracy = model.evaluate(x_test, y_test_oh, verbose=0)

print(f"\nMNIST Test Accuracy: {accuracy * 100:.2f}%")

TensorFlow version: 2.18.0

Model loaded successfully.


Model: "functional_3"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ input_layer_2       │ (None, 28, 28, 1) │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ augmentation        │ (None, 28, 28, 1) │          0 │ input_layer_2[0]… │
│ (Sequential)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2d_15 (Conv2D)  │ (None, 28, 28,    │        288 │ augmentation[0][… │
│                     │ 32)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ batch_normalizatio… │ (None, 28, 28,    │        128 │ conv2d_15[0][0]   │
│ (BatchNormalizatio… │ 32)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ activation_13       │ (None, 28, 28,    │          0 │ batch_normalizat… │
│ (Activation)        │ 32)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2d_16 (Conv2D)  │ (None, 28, 28,    │      9,216 │ activation_13[0]… │
│                     │ 32)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ batch_normalizatio… │ (None, 28, 28,    │        128 │ conv2d_16[0][0]   │
│ (BatchNormalizatio… │ 32)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ activation_14       │ (None, 28, 28,    │          0 │ batch_normalizat… │
│ (Activation)        │ 32)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2d_17 (Conv2D)  │ (None, 28, 28,    │      9,216 │ activation_14[0]… │
│                     │ 32)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ batch_normalizatio… │ (None, 28, 28,    │        128 │ conv2d_17[0][0]   │
│ (BatchNormalizatio… │ 32)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ add_6 (Add)         │ (None, 28, 28,    │          0 │ batch_normalizat… │
│                     │ 32)               │            │ activation_13[0]… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ activation_15       │ (None, 28, 28,    │          0 │ add_6[0][0]       │
│ (Activation)        │ 32)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2d_18 (Conv2D)  │ (None, 28, 28,    │      9,216 │ activation_15[0]… │
│                     │ 32)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ batch_normalizatio… │ (None, 28, 28,    │        128 │ conv2d_18[0][0]   │
│ (BatchNormalizatio… │ 32)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ activation_16       │ (None, 28, 28,    │          0 │ batch_normalizat… │
│ (Activation)        │ 32)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2d_19 (Conv2D)  │ (None, 28, 28,    │      9,216 │ activation_16[0]… │
│                     │ 32)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ batch_normalizatio… │ (None, 28, 28,    │        128 │ conv2d_19[0][0] 

 Total params: 698,283 (2.66 MB)

 Trainable params: 696,042 (2.66 MB)

 Non-trainable params: 2,240 (8.75 KB)

 Optimizer params: 1 (8.00 B)


MNIST Test Accuracy: 99.64%
